# Swiss Legal Retrieval — Finalist AUDIT Notebook

Competition: [LLM Agentic Legal Information Retrieval](https://www.kaggle.com/competitions/llm-agentic-legal-information-retrieval)

Team: WBF_USA_NYC · Final-selection date: 2026-05-19

## What this notebook does

**This is the AUDIT notebook** — it reproduces, byte-identical, any of the three submissions selected for private-leaderboard judging. Each output is SHA256-verified against the canonical hash recorded in `PRIZE_REPRO_DO_NOT_DELETE.md` and `scripts/final_submission_lock.py`. No network calls, no LLM endpoints required.

**It is NOT the prize-qualification notebook.** The competition is a code competition; the host can re-evaluate the prize-qualification notebook on unseen queries (HIDDEN.csv), and a notebook that returns the locked CSV from disk regardless of input will fail that test. The prize-qualification notebook is an offline pipeline that generalizes — see `CODEX_OFFLINE_NOTEBOOK_HANDOVER_2026-05-19.md`.

## Modes

Set `SUBMISSION_MODE` (env var) or change the `MODE` constant in the next cell:

| Mode | File | Public LB | Kaggle ref | Role |
|---|---|---|---|---|
| `intersect_bold7h_33028` | `intersect_bold7h_33028.csv` | 0.33028 | 52819486 | Private precision/intersection hedge |
| `public_peak_33438` | `public_peak_33438.csv` | 0.33438 | 52758343 | Public-LB peak safety net |
| `fusion_samesrc03_32274` | `fusion_samesrc03_32274.csv` | 0.32274 | 52596721 | Recall/diversity hedge |

## Required inputs (when running on Kaggle)

Upload the project's reproducibility dataset so it mounts at one of:
- `/kaggle/input/swiss-legal-finalists-2026-05-19/`
- `/kaggle/input/datasets/wbfranci/swiss-legal-finalists-2026-05-19/`

The dataset must contain the three CSVs named exactly as in the table above. Each file is independently SHA256-checked.

## Methodology

The retrieval/judging/perturbation pipeline that originally produced these CSVs is documented in `SOLUTION_WRITEUP.md` (this repo) and implemented across `pipeline_v11.py`, `run_v11_staged.py`, `scripts/winner_localperturb_search.py`, and `notebooks/swiss_submission_v12.py`. This notebook is the AUDIT path — deterministic regeneration from the locked artifacts.

In [ ]:
import os

# Choose which finalist to materialize. Override at runtime with the SUBMISSION_MODE env var.
MODE = os.environ.get("SUBMISSION_MODE", "intersect_bold7h_33028")
print(f"Reproducing mode: {MODE}")

In [ ]:
import hashlib
import os
import shutil
from pathlib import Path

LOCKED_PAYLOADS = {
    "intersect_bold7h_33028": {
        "kaggle_dataset_file": "intersect_bold7h_33028.csv",
        "sha256": "542b40471aec01c53a891cccab44e8b3495cf6bb06e6c1a521fe88da61afd8ca",
        "public_score": "0.33028",
        "kaggle_ref": "52819486",
        "role": "Private precision/intersection hedge",
    },
    "public_peak_33438": {
        "kaggle_dataset_file": "public_peak_33438.csv",
        "sha256": "89acefcdc37eeaf7d08b99559427a5167e7997cf5304a02278c7f09e27c85b9b",
        "public_score": "0.33438",
        "kaggle_ref": "52758343",
        "role": "Public-LB peak safety net",
    },
    "fusion_samesrc03_32274": {
        "kaggle_dataset_file": "fusion_samesrc03_32274.csv",
        "sha256": "163f0bba09ca6e07d648a62360abb09bb63f45f227e3e421b3f569b84225d5d2",
        "public_score": "0.32274",
        "kaggle_ref": "52596721",
        "role": "Recall/diversity hedge",
    },
}

info = LOCKED_PAYLOADS[MODE]

def find_input(filename):
    candidates = [
        Path("/kaggle/input/swiss-legal-finalists-2026-05-19") / filename,
        Path("/kaggle/input/datasets/wbfranci/swiss-legal-finalists-2026-05-19") / filename,
    ]
    for c in candidates:
        if c.exists():
            return c
    for root, _dirs, files in os.walk("/kaggle/input"):
        if filename in files:
            return Path(root) / filename
    raise FileNotFoundError(f"Could not find {filename} in /kaggle/input. "
                            "Upload the reproducibility dataset first.")

source = find_input(info["kaggle_dataset_file"])
print(f"Source: {source}")

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

actual = sha256(source)
if actual != info["sha256"]:
    raise SystemExit(f"SHA256 mismatch!\n  expected: {info['sha256']}\n  actual:   {actual}")
print(f"SHA256 OK: {actual}")

In [ ]:
out_path = Path("/kaggle/working/submission.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(source, out_path)

copied = sha256(out_path)
assert copied == info["sha256"], "Post-copy hash drift"

print("=" * 70)
print(f"mode           : {MODE}")
print(f"role           : {info['role']}")
print(f"public_score   : {info['public_score']}")
print(f"kaggle_ref     : {info['kaggle_ref']}")
print(f"output         : {out_path}")
print(f"sha256         : {copied}")
print("=" * 70)
print("OK — byte-identical to locked payload.")

In [ ]:
# Sanity check: 40 rows, correct header, query_ids test_001..test_040.
import csv
with open(out_path, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)
assert header == ["query_id", "predicted_citations"], header
assert len(rows) == 40, len(rows)
ids = sorted(r[0] for r in rows)
expected = sorted(f"test_{i:03d}" for i in range(1, 41))
assert ids == expected
print(f"Shape OK: 40 rows, header matches, query_ids test_001..test_040.")
print(f"Sample row: {rows[0]}")